<a href="https://colab.research.google.com/github/atanuduttagupta/ai-learning-journey/blob/Day-52-Simple-SLM/Day_52_SLM_Handson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run this first to install the Hugging Face transformers ecosystem, accelerate (for handling device hardware), and bitsandbytes (for shrinking the model size).

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch

This step breaks down how we safely load a 3.8-billion-parameter model into memory by using 4-bit quantization (compression)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The Hugging Face registry ID for Microsoft's Phi-3 model
model_id = "microsoft/Phi-3-mini-4k-instruct"

# ==========================================
# STEP A: COMPRESSION CONFIGURATION (4-BIT)
# ==========================================
# Raw LLM weights are huge. This configuration compresses the weights
# from 16-bit down to 4-bit numbers so it fits in regular computer RAM/VRAM.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,          # Enable 4-bit compression
    bnb_4bit_quant_type="nf4",  # 'NormalFloat 4' - a highly accurate data type for LLMs
    bnb_4bit_compute_dtype=torch.float16 # The data type used during actual math calculations
)

# ==========================================
# STEP B: LOAD THE TOKENIZER
# ==========================================
# Text cannot be read directly by neural networks. The tokenizer converts
# raw strings into math-friendly integers called "Tokens".
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# ==========================================
# STEP C: LOAD THE ACTUAL MODEL
# ==========================================
print("Loading model (this might take a minute)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # Apply our 4-bit compression settings
    device_map="auto",              # Automatically shifts data between CPU and GPU if available
    trust_remote_code=False          # Allows Hugging Face to execute model-specific code locally
)

print("Model successfully loaded!")

This function exposes the core pipeline lifecycle of an LLM: Formatting $\rightarrow$ Tokenizing $\rightarrow$ Matrix Multiplication (Generating) $\rightarrow$ Detokenizing.

In [ ]:
def ask_phi(prompt):
    # 1. Structure the conversation text
    messages = [
        {"role": "user", "content": prompt}
    ]

    # 2. Convert structure into tokens
    # We add return_dict=True to explicitly tell the tokenizer how to pack the outputs,
    # then we unpack it cleanly.
    tokenized_output = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,            # Explicitly return as a dictionary format
        return_tensors="pt"
    )

    # Extract the actual input_ids tensor array and push it to the GPU/CPU device
    input_ids = tokenized_output["input_ids"].to(model.device)
    attention_mask = tokenized_output["attention_mask"].to(model.device) # <-- ADDED THIS LINE

    # 3. Model Inference Execution
    # Pass the input_ids directly into the generator
    outputs = model.generate(
        input_ids,                   # Passing the raw tensor instead of the dictionary wrapper
        attention_mask=attention_mask,  # <-- PASSED MASK HERE to tell the model what to ignore
        max_new_tokens=800,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    # 4. Slicing out ONLY the newly generated answer tokens
    # input_ids.shape[-1] now works flawlessly because input_ids is a pure PyTorch tensor!
    new_tokens_only = outputs[0][input_ids.shape[-1]:]

    # 5. Detokenization back to text
    decoded_output = tokenizer.decode(new_tokens_only, skip_special_tokens=True)
    return decoded_output.strip()

Now execute the code blocks sequentially and test the core workflow loop!

In [ ]:
# Call the function and print the processed text output
response = ask_phi("How to make Noodles?")
print(response)